In [0]:
%run ../Copy-Datasets

In [0]:
# %run ../Copy-Datasets

current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
dataset_bookstore = f"/Volumes/{current_catalog}/bookstore_eng_pro/dataset"
checkpoint_path = f"/Volumes/{current_catalog}/bookstore_eng_pro/checkpoints"
print(dataset_bookstore)

In [0]:
from pyspark.sql import functions as F

schema = "customer_id STRING, email STRING, first_name STRING, last_name STRING, gender STRING, street STRING, city STRING, country_code STRING, row_status STRING, row_time timestamp"

(spark.readStream
        .table("bronze")
        .filter("topic = 'customers'")
        .select(F.from_json(F.col("value").cast("string"), schema).alias("v"))
        .select("v.*", F.col('v.row_time').alias("request_timestamp"))
        .filter("row_status = 'delete'")
        .select("customer_id", "request_timestamp",
                F.date_add("request_timestamp", 30).alias("deadline"), 
                F.lit("requested").alias("status"))
    .writeStream
        .outputMode("append")
        .option("checkpointLocation", f"{bookstore.checkpoint_path}/delete_requests")
        .trigger(availableNow=True)
        .table("delete_requests")
)

In [0]:
%sql
SELECT * FROM delete_requests

In [0]:
%sql
DELETE FROM customers_silver
WHERE customer_id IN (SELECT customer_id FROM delete_requests WHERE status = 'requested')

In [0]:
deleteDF = (spark.readStream
                 .format("delta")
                 .option("readChangeFeed", "true")
                 .option("startingVersion", 2)
                 .table("customers_silver"))

In [0]:
def process_deletes(microBatchDF, batchId):
    
    (microBatchDF
        .filter("_change_type = 'delete'")
        .createOrReplaceTempView("deletes"))

    microBatchDF.sparkSession.sql("""
        DELETE FROM customers_orders
        WHERE customer_id IN (SELECT customer_id FROM deletes)
    """)
    
    microBatchDF.sparkSession.sql("""
        MERGE INTO delete_requests r
        USING deletes d
        ON d.customer_id = r.customer_id
        WHEN MATCHED
          THEN UPDATE SET status = "deleted"
    """)

In [0]:
(deleteDF.writeStream
         .foreachBatch(process_deletes)
         .option("checkpointLocation", f"{bookstore.checkpoint_path}/deletes")
         .trigger(availableNow=True)
         .start())

In [0]:
%sql
SELECT * FROM delete_requests

In [0]:
%sql
DESCRIBE HISTORY customers_orders

In [0]:
%sql
SELECT * FROM customers_orders@v6
EXCEPT
SELECT * FROM customers_orders

In [0]:
df = (spark.read
           .option("readChangeFeed", "true")
           .option("startingVersion", 2)
           .table("customers_silver")
           .filter("_change_type = 'delete'"))
display(df)